# 04 · MobileNetV2 experiments  *(Onkar - part 2b)*

Each experiment changes **one thing** relative to the default `mobilenet_v2` run and asks one question. All are judged on the **validation** split only (the test split is not used).

| Run | Change | Question |
|---|---|---|
| `head_only` | no fine-tuning (stage 1 only) | is fine-tuning worth the extra training? |
| `ft_top20` / `ft_top80` | unfreeze 20 / 80 layers (default 40) | how deep should fine-tuning go? |
| `ft_lr_1e-5` / `ft_lr_5e-5` | fine-tuning lr (default 2e-5) | is the default learning rate good? |
| `no_class_weights` | classes weighted equally | do class weights help the weak classes? |
| `no_augmentation` | no random augmentation | how much does augmentation matter? |
| `dropout_0.5` | dropout 0.5 (default 0.3) | does stronger regularisation help? |
| `alpha_0.75` | 25 % slimmer network | accuracy lost for a smaller, faster model? |
| `size_160` | 160×160 input (default 224) | accuracy lost for cheaper inference? |

**Run order / cost:** group A reuses the default stage-1 model (cheap). Group B needs its own stage 1 (about as long as the default run each). Run group A first, then as many of group B as time allows.

In [ ]:
# ============ SETUP - run this first after every fresh Colab runtime ============
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # e.g. "https://github.com/<your-username>/fruit-freshness-classifier.git"
                # leave "" if you uploaded/unzipped the project folder yourself

IN_COLAB = "google.colab" in sys.modules
PROJECT_DIR = Path("/content/fruit-freshness-classifier") if IN_COLAB else Path.cwd().parent
if not (PROJECT_DIR / "src").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        raise SystemExit(f"Project folder not found at {PROJECT_DIR}. Set REPO_URL above or upload the project there.")
elif REPO_URL and IN_COLAB:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

if os.getenv("FRUIT_SKIP_PIP") != "1":     # only kagglehub is needed; Colab already has TensorFlow etc.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import json
import matplotlib.pyplot as plt
import pandas as pd
from fruitfresh import config
from fruitfresh.colab import setup_workspace, gpu_summary

WORK_DIR = setup_workspace()      # Colab: Google Drive (survives runtime resets)
SEED = config.SEED
print(gpu_summary())
print("Project folder :", PROJECT_DIR)
print("Work folder    :", WORK_DIR, "(models, histories, results are saved here)")

In [ ]:
from fruitfresh.data_index import get_dataset_root
from fruitfresh.splits import load_split

DATASET_ROOT = get_dataset_root()
SPLIT_DIR = WORK_DIR / "data" / "splits"
if not SPLIT_DIR.exists():
    SPLIT_DIR = PROJECT_DIR / "data" / "splits"      # the committed copy of the team split
train_df, val_df, _ = load_split(SPLIT_DIR)           # the TEST split is deliberately ignored (`_`)
print(f"train {len(train_df)} images / {train_df.source_id.nunique()} photos | "
      f"validation {len(val_df)} images / {val_df.source_id.nunique()} photos")
print("Split folder:", SPLIT_DIR)

In [ ]:
PRETRAINED = os.getenv("FRUIT_NO_PRETRAINED", "0") != "1"
GROUP_A = ["head_only", "ft_top20", "ft_top80", "ft_lr_1e-5", "ft_lr_5e-5"]            # reuse stage 1: cheap
GROUP_B = ["no_class_weights", "no_augmentation", "dropout_0.5", "alpha_0.75", "size_160"]   # own stage 1: slower
EXPERIMENTS = GROUP_A            # <- later: GROUP_A + GROUP_B (finished runs are skipped automatically)
FORCE = False
print("Will run:", EXPERIMENTS)

## 1 · Run the experiments
The default `mobilenet_v2` run from notebook 03 is reused if it exists.

In [ ]:
from fruitfresh.experiments import run_mobilenet_experiments
infos = run_mobilenet_experiments(EXPERIMENTS, train_df, val_df, DATASET_ROOT, WORK_DIR, SEED,
                                  pretrained=PRETRAINED, force=FORCE, verbose=2)
print("finished:", list(infos))

## 2 · Compare on validation

In [ ]:
from fruitfresh.experiments import summarize_runs, plot_experiments
from fruitfresh.train import run_dir_for
finished = [r for r in ["mobilenet_v2"] + GROUP_A + GROUP_B if (run_dir_for(WORK_DIR, r, SEED) / "model.keras").exists()]
table = summarize_runs(WORK_DIR, finished, SEED, val_df, DATASET_ROOT)
display(table.sort_values("val_macro_f1", ascending=False).reset_index(drop=True))
fig = plot_experiments(table); plt.show()

## 3 · How to interpret (important)
* The validation set contains only a few dozen distinct photos per class, so **differences below ~1 point of macro-F1 are noise** - do not write "X is better" for such gaps.
* Compare `val_loss` as well: it is smoother than accuracy.
* `no_augmentation` / `no_class_weights` often cost little on a clean dataset: report that honestly.
* `alpha_0.75` and `size_160` answer the **limited-hardware** question: report the accuracy lost *and* the size/time saved (`params (M)`, `train_min`).
* One run per setting (one seed) is a limitation. If time allows, repeat the 2-3 most interesting experiments with `SEED + 1`.

In [ ]:
from fruitfresh.experiments import pick_best
best = pick_best(table)
print("Best MobileNetV2 run on validation:", best["run"])
print(best.to_string())
table.to_csv(WORK_DIR / "experiments.csv", index=False)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
table.to_csv(PROJECT_DIR / "results" / "mobilenetv2_experiments_validation.csv", index=False)
print("saved experiments.csv (Drive) and results/mobilenetv2_experiments_validation.csv (repo, commit it)")

## 4 · Your findings (fill in from the table - one line per experiment)
| Run | Result vs default (val macro-F1, val loss) | Conclusion |
|---|---|---|
| head_only | | |
| ft_top20 / ft_top80 | | |
| ft_lr_* | | |
| no_class_weights | | |
| no_augmentation | | |
| dropout_0.5 | | |
| alpha_0.75 | | |
| size_160 | | |

## Checklist before notebook 05
- [ ] at least group A finished; group B as time allows
- [ ] `experiments.csv` saved; findings table filled in with numbers copied from the output
- [ ] the winner was chosen on **validation**, with the rule in `pick_best`